# Practical Assignment 1: Word Embeddings (Word2Vec)

**Course:** Deep Learning for Natural Language Processing (NLP)  
**Academic Semester:** 2026/1  
**Author:** Gustavo Freitas Cunha  
**Subject:** Hyperparameter Exploration and Analogy Evaluation in Word2Vec  

---

> **AI Disclosure and Academic Integrity Statement**  
> In accordance with institutional guidelines on the ethical use of Artificial Intelligence (https://www.ufmg.br/ia/), this deliverable documents the utilization of AI assistance:  
>  
> - **Agents Utilized:** Google Antigravity (agentic assistant powered by Gemini).  
> - **Scope of AI Assistance:** Used strictly for code boilerplate generation, documentation scaffolding, and syntax formatting.  
> - **Student Authorship and Oversight:** All conceptual formulations, hyperparameter decisions, experimental workflows, and analytical interpretations were designed and authored directly by me.  
> - **Validation:** Every pipeline component and result was reviewed, compiled, and validated by me in the runtime environment.


# 1. Setup and Environment

In this section, I install required dependencies, centralize scientific library imports, and configure a fixed random seed of 42 for experimental reproducibility.


In [24]:
#CELL 1.1 - INSTALL REQUIRED DEPENDENCIES
!pip install -q gensim numpy scipy matplotlib seaborn pandas

In [25]:
#CELL 1.2 - IMPORT DEPENDENCIES AND CONFIGURE REPRODUCIBILITY ENVIRONMENT
import os
import sys
import gc
import time
import zipfile
import urllib.request
import platform
import subprocess
import resource
import logging
from datetime import datetime, timedelta
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import pandas as pd
import scipy.spatial.distance as scipy_dist
import matplotlib.pyplot as plt
import seaborn as sns

import gensim
from gensim.models import Word2Vec
from gensim.models.word2vec import Text8Corpus

#configure logging and visualization aesthetics
logging.basicConfig(
    format="%(asctime)s : %(levelname)s : %(message)s",
    level=logging.INFO
)
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120

#fix deterministic seed
GLOBAL_RANDOM_SEED = 42
np.random.seed(GLOBAL_RANDOM_SEED)

#configure multiprocessing parallelism based on available cpu cores
NUM_WORKERS = max(1, os.cpu_count() or 2)

#hardware profiling helper utilities
def get_hardware_profile() -> Dict[str, Any]:
    cpu_model = "Unknown CPU"
    try:
        with open("/proc/cpuinfo", "r") as f:
            for line in f:
                if "model name" in line:
                    cpu_model = line.split(":")[1].strip()
                    break
    except Exception:
        cpu_model = platform.processor() or "Unknown CPU"
        
    total_ram_gb = 0.0
    try:
        with open("/proc/meminfo", "r") as f:
            for line in f:
                if "MemTotal" in line:
                    total_ram_gb = round(int(line.split()[1]) / (1024 * 1024), 2)
                    break
    except Exception:
        pass
        
    gpu_info = "None (CPU runtime)"
    try:
        res = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True,
            text=True
        )
        if res.returncode == 0 and res.stdout.strip():
            gpu_info = res.stdout.strip().replace("\n", "; ")
    except Exception:
        pass
        
    return {
        "cpu_model": cpu_model,
        "cpu_cores": os.cpu_count() or 1,
        "total_ram_gb": total_ram_gb,
        "gpu_info": gpu_info,
        "workers_configured": NUM_WORKERS
    }

env_profile = get_hardware_profile()
print(f"Environment ready. Gensim: {gensim.__version__} | Pandas: {pd.__version__}")
print(f"Hardware Profile: {env_profile['cpu_model']} ({env_profile['cpu_cores']} cores) | RAM: {env_profile['total_ram_gb']} GB | GPU: {env_profile['gpu_info']} | Parallel Workers: {NUM_WORKERS}")


Environment ready. Gensim: 4.4.0 | Pandas: 2.2.3
Hardware Profile: Intel(R) Xeon(R) CPU @ 2.20GHz (2 cores) | RAM: 12.67 GB | GPU: None (CPU runtime) | Parallel Workers: 2


# 2. Data Acquisition, Preprocessing and Corpus Preparation

In this section, I load and prepare the experimental datasets:
- **`text8` Corpus:** 100 MB Wikipedia sample (~17M tokens) streamed from disk via `Text8Corpus`.
- **`questions-words.txt` Dataset:** 19,544 analogy quadruplets across semantic and syntactic categories.

### Preprocessing and Vocabulary Strategy
- **Normalization and Deduplication:** I normalize all analogy tokens to lowercase to match `text8` and remove category-level duplicates.
- **Vocabulary Threshold (`min_count=5`):** I retain tokens with frequency at least 5 (71,290 words), pruning infrequent tokens that lack sufficient distributional context for gradient stabilization. I audit frequency tiers (hapax legomena with frequency 1 vs. rare words with frequency 2 to 4) for diagnostic profiling.
- **Benchmark Integrity:** I preserve all analogy quadruplets intact without pre-filtering, handling out-of-vocabulary instances dynamically during evaluation.


In [26]:
#CELL 2.1 - CONFIGURE DATA PATHS AND DOWNLOAD BENCHMARK DATASETS
#resolve data directory across local and cloud environments
if Path("data").exists():
    DATA_DIR = Path("data")
elif Path("../data").exists():
    DATA_DIR = Path("../data")
elif Path("TP1/data").exists():
    DATA_DIR = Path("TP1/data")
else:
    DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TEXT8_EXTRACTED_PATH = DATA_DIR / "text8"
TEXT8_ZIP_PATH = DATA_DIR / "text8.zip"
QUESTIONS_FILE_PATH = DATA_DIR / "questions-words.txt"

TEXT8_URL = "http://mattmahoney.net/dc/text8.zip"
QUESTIONS_URL = "https://raw.githubusercontent.com/nicholas-leonard/word2vec/master/questions-words.txt"

#check existing files to prevent redundant downloads and duplicates
if TEXT8_EXTRACTED_PATH.exists():
    print(f"Verified existing text8 corpus at: {TEXT8_EXTRACTED_PATH.resolve()} (skipping download to prevent duplicates)")
else:
    print(f"Downloading text8 archive from {TEXT8_URL}...")
    urllib.request.urlretrieve(TEXT8_URL, TEXT8_ZIP_PATH)
    print("Extracting text8 corpus...")
    with zipfile.ZipFile(TEXT8_ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)
    #remove temporary archive to avoid duplicates and save disk space
    if TEXT8_ZIP_PATH.exists():
        TEXT8_ZIP_PATH.unlink()
    print("Extraction complete and temporary archive removed.")

#download analogy benchmark only if missing
if QUESTIONS_FILE_PATH.exists():
    print(f"Verified existing analogy dataset at: {QUESTIONS_FILE_PATH.resolve()} (skipping download to prevent duplicates)")
else:
    print(f"Downloading questions-words.txt from {QUESTIONS_URL}...")
    urllib.request.urlretrieve(QUESTIONS_URL, QUESTIONS_FILE_PATH)
    print("Download complete.")

print(f"\nData verification summary:")
print(f"- text8 corpus: {TEXT8_EXTRACTED_PATH.stat().st_size / (1024 * 1024):.2f} MB")
print(f"- questions-words dataset: {QUESTIONS_FILE_PATH.stat().st_size / 1024:.2f} KB")

Verified existing text8 corpus at: /content/data/text8 (skipping download to prevent duplicates)
Verified existing analogy dataset at: /content/data/questions-words.txt (skipping download to prevent duplicates)

Data verification summary:
- text8 corpus: 95.37 MB
- questions-words dataset: 589.80 KB


In [27]:
#CELL 2.2 - PARSE AND CLEAN ANALOGY BENCHMARK INTO STRUCTURED DATAFRAME
#parse raw text format into structured records
raw_records: List[Dict[str, str]] = []
active_category: Optional[str] = None

with open(QUESTIONS_FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        if line.startswith(":"):
            active_category = line[1:].strip()
        else:
            tokens = line.lower().split()
            #verify quadruplet token integrity
            if len(tokens) == 4 and active_category is not None:
                section_type = "syntactic" if active_category.startswith("gram") else "semantic"
                raw_records.append({
                    "category": active_category,
                    "section_type": section_type,
                    "word_a": tokens[0],
                    "word_b": tokens[1],
                    "word_c": tokens[2],
                    "word_d": tokens[3]
                })

df_analogies_raw = pd.DataFrame(raw_records)

#filter out duplicate rows within the same category
initial_count = len(df_analogies_raw)
df_analogies = df_analogies_raw.drop_duplicates(
    subset=["category", "word_a", "word_b", "word_c", "word_d"]
).reset_index(drop=True)
duplicates_removed = initial_count - len(df_analogies)

print(f"Analogy parsing and cleaning completed:")
print(f"- Total records parsed: {initial_count}")
print(f"- Duplicates removed: {duplicates_removed}")
print(f"- Clean unique quadruplets: {len(df_analogies)}")
print(f"- Section distribution:\n{df_analogies['section_type'].value_counts()}")
display(df_analogies.head(5))

Analogy parsing and cleaning completed:
- Total records parsed: 19544
- Duplicates removed: 0
- Clean unique quadruplets: 19544
- Section distribution:
section_type
syntactic    10675
semantic      8869
Name: count, dtype: int64


,category,section_type,word_a,word_b,word_c,word_d
0,capital-common-countries,semantic,athens,greece,baghdad,iraq
1,capital-common-countries,semantic,athens,greece,bangkok,thailand
2,capital-common-countries,semantic,athens,greece,beijing,china
3,capital-common-countries,semantic,athens,greece,berlin,germany
4,capital-common-countries,semantic,athens,greece,bern,switzerland


In [28]:
#CELL 2.3 - INITIALIZE STREAMING CORPUS AND CONSTRUCT VOCABULARY FREQUENCY PROFILE
#stream corpus from disk without full in-memory loading
corpus = Text8Corpus(str(TEXT8_EXTRACTED_PATH))

#profile word frequency distribution across corpus
vocab_counter: Counter = Counter()
total_corpus_tokens = 0

with open(TEXT8_EXTRACTED_PATH, "r", encoding="utf-8") as f:
    for line in f:
        tokens = line.split()
        total_corpus_tokens += len(tokens)
        vocab_counter.update(tokens)

unique_raw_words = len(vocab_counter)
#quantify frequency distribution across tiers
hapax_count = sum(1 for c in vocab_counter.values() if c == 1)
rare_count = sum(1 for c in vocab_counter.values() if 1 < c < 5)
MIN_WORD_COUNT = 5
vocab_filtered: Set[str] = {w for w, c in vocab_counter.items() if c >= MIN_WORD_COUNT}

print(f"Corpus frequency profiling completed:")
print(f"- Total running tokens: {total_corpus_tokens:,}")
print(f"- Total unique vocabulary words: {unique_raw_words:,}")
print(f"- True hapax legomena (frequency == 1): {hapax_count:,} ({hapax_count / unique_raw_words * 100:.2f}%)")
print(f"- Rare words (1 < frequency < 5): {rare_count:,} ({rare_count / unique_raw_words * 100:.2f}%)")
print(f"- Retained vocabulary (frequency >= {MIN_WORD_COUNT}): {len(vocab_filtered):,} ({len(vocab_filtered) / unique_raw_words * 100:.2f}%)")


Corpus frequency profiling completed:
- Total running tokens: 17,005,207
- Total unique vocabulary words: 253,854
- True hapax legomena (frequency == 1): 118,519 (46.69%)
- Rare words (1 < frequency < 5): 64,045 (25.23%)
- Retained vocabulary (frequency >= 5): 71,290 (28.08%)


In [29]:
#CELL 2.4 - AUDIT VOCABULARY OVERLAP AND TAG IN-VOCABULARY ANALOGIES
#tag analogy quadruplets where all four words are present in filtered vocabulary
def verify_quadruplet_in_vocab(row: pd.Series, vocab: Set[str]) -> bool:
    return (
        row["word_a"] in vocab and
        row["word_b"] in vocab and
        row["word_c"] in vocab and
        row["word_d"] in vocab
    )

#audit coverage while preserving the entire dataset intact
df_analogies["in_vocab"] = df_analogies.apply(
    lambda r: verify_quadruplet_in_vocab(r, vocab_filtered),
    axis=1
)

valid_analogies_count = df_analogies["in_vocab"].sum()
total_analogies = len(df_analogies)
coverage_pct = (valid_analogies_count / total_analogies) * 100

print(f"Vocabulary audit on analogy benchmark (min_count={MIN_WORD_COUNT}):")
print(f"- Fully covered quadruplets (ready for evaluation): {valid_analogies_count:,} ({coverage_pct:.2f}%)")
print(f"- Quadruplets with OOV tokens: {total_analogies - valid_analogies_count:,} ({100 - coverage_pct:.2f}%)")
print(f"- Total intact benchmark quadruplets preserved: {total_analogies:,}")

#breakdown coverage by category
category_audit = df_analogies.groupby(["section_type", "category"])["in_vocab"].agg(
    total_quadruplets="count",
    covered_quadruplets="sum",
    coverage_ratio="mean"
).reset_index()
category_audit["coverage_pct"] = category_audit["coverage_ratio"] * 100

display(category_audit.sort_values(by="coverage_pct", ascending=False))


Vocabulary audit on analogy benchmark (min_count=5):
- Fully covered quadruplets (ready for evaluation): 17,827 (91.21%)
- Quadruplets with OOV tokens: 1,717 (8.79%)
- Total intact benchmark quadruplets preserved: 19,544


,section_type,category,total_quadruplets,covered_quadruplets,coverage_ratio,coverage_pct
0,semantic,capital-common-countries,506,506,1.000000,100.000000
5,syntactic,gram1-adjective-to-adverb,992,992,1.000000,100.000000
11,syntactic,gram7-past-tense,1560,1560,1.000000,100.000000
12,syntactic,gram8-plural,1332,1332,1.000000,100.000000
9,syntactic,gram5-present-participle,1056,1056,1.000000,100.000000
7,syntactic,gram3-comparative,1332,1332,1.000000,100.000000
13,syntactic,gram9-plural-verbs,870,870,1.000000,100.000000
10,syntactic,gram6-nationality-adjective,1599,1521,0.951220,95.121951
2,semantic,city-in-state,2467,2330,0.944467,94.446696
6,syntactic,gram2-opposite,812,756,0.931034,93.103448


# 3. Word2Vec Architecture and Modular Training Pipeline

In this section, I implement a modular training pipeline using Gensim's Word2Vec architecture.

### Architectural Framework Justification: Gensim vs. Standalone C Repository
The assignment handout suggests using the standalone C repository by Nicholas Leonard (https://github.com/nicholas-leonard/word2vec), an adaptation of Mikolov's original C code. In this project, I adopt the standard Python architecture library **Gensim** (`gensim.models.Word2Vec`) based on the following technical justifications:
1. **Algorithmic Equivalence and Fidelity:** Gensim implements Mikolov's exact Word2Vec formulations (Continuous Bag-of-Words and Skip-gram with negative sampling) directly in Cython backed by low-level BLAS routines. It maintains identical mathematical mechanics and computational throughput to the reference C implementation.
2. **Native Pipeline Integration:** Running Word2Vec natively within Python eliminates external C compilation dependencies, shell-level subprocess overhead, and disk serialization of intermediate vector files. This allows seamless integration with streaming corpus iterators (`Text8Corpus`), direct in-memory array extraction for NumPy algebraic evaluations, and resilient checkpointing.

### Architectural Mechanics: CBOW vs. Skip-gram
- **CBOW (`sg=0`):** Continuous Bag-of-Words averages the representations of surrounding context words to predict the center target word. It trains faster and captures frequent syntactic regularities effectively.
- **Skip-gram (`sg=1`):** Skip-gram uses the center word to predict each surrounding context word independently. It gives higher statistical weight to rare word contexts and captures fine-grained semantic nuances, at the cost of higher training time.

### Systematic Hyperparameter Grid Search (54 Configurations)

To evaluate the impact of architecture and training hyperparameters on semantic and syntactic representations, I perform a full Cartesian grid search across four operational axes:

| Hyperparameter | Code Parameter | Explored Values | Count | Search Purpose and Representation Role |
| :--- | :--- | :--- | :--- | :--- |
| **Architecture** | `sg` | `0` (CBOW), `1` (Skip-gram) | 2 | Context aggregation vs. target-to-context prediction |
| **Context Window** | `window` | `2`, `5`, `10` words | 3 | Local syntactic dependencies ($w=2$) to broad topical semantics ($w=10$) |
| **Vector Dimension** | `vector_size` | `50`, `100`, `300` dimensions | 3 | Latent geometric capacity: compact ($d=50$) to high-capacity ($d=300$) |
| **Training Epochs** | `epochs` | `1`, `5`, `10` passes | 3 | Optimization convergence: single pass ($e=1$) to extended convergence ($e=10$) |

#### Justification of Explored Values Grounded in Literature
- **Architecture (`sg` $\in$ {0, 1}):** Mikolov et al. (2013a, 2013b) established Continuous Bag-of-Words (CBOW) and Skip-gram as complementary training objectives. CBOW optimizes prediction of the target word from averaged context representations, proving computationally faster and effective for syntactic relations. Skip-gram predicts surrounding words independently, which Mikolov et al. (2013b) demonstrated yields superior representations for rare words and semantic analogies. I evaluate both architectures under identical conditions to assess this trade-off.
- **Context Window (`window` $\in$ {2, 5, 10}):** Levy and Goldberg (2014) demonstrated that context window size dictates the qualitative nature of learned embeddings: small windows ($w=2$) constrain context to immediate syntactic dependencies and functional equivalents (part-of-speech, inflections), whereas larger windows ($w \ge 5, 10$) capture broad thematic and topical relatedness. Mikolov et al. (2013a) recommended $w=5$ for CBOW and $w=10$ for Skip-gram. I select $w \in \{2, 5, 10\}$ to capture this exact spectrum from localized syntax to broad topical semantics.
- **Vector Dimension (`vector_size` $\in$ {50, 100, 300}):** Turian et al. (2010) and Mikolov et al. (2013a) demonstrated that representation accuracy scales logarithmically with dimensionality before plateauing. Yin and Shen (2018) formalized this dimensionality trade-off, showing that low dimensions ($d=50$) induce representational bias through semantic bottlenecking, while excessive dimensions risk variance and noise on medium-sized corpora. I select $d=50$ as a low-capacity probe, $d=100$ as the standard baseline for moderate corpora, and $d=300$ as the landmark dimensionality established by Mikolov et al. (2013b) in the Google News embeddings.
- **Training Epochs (`epochs` $\in$ {1, 5, 10}):** While Mikolov et al. (2013a) trained on massive multi-billion-token corpora using a single pass ($e=1$), empirical studies on smaller corpora like text8 (~17M tokens) by Lai et al. (2016) and standard Gensim implementations show that multiple stochastic gradient updates are essential for loss convergence. I evaluate $e=1$ (single-pass baseline), $e=5$ (standard convergence default), and $e=10$ (extended optimization) to investigate whether repeated passes yield diminishing returns or overfit the analogy task.

#### Grid Dimensions and Model Comparison Scope
- **Discrete Parameter Options:**
  - Architecture $\in$ {CBOW (`sg=0`), Skip-gram (`sg=1`)} (2 options)
  - Context Window $\in$ {2, 5, 10} words (3 options)
  - Embedding Dimension $\in$ {50, 100, 300} dimensions (3 options)
  - Training Epochs $\in$ {1, 5, 10} passes (3 options)
- **Total Models Resulting from Grid Search:**
  $$\text{Total Models} = 2 \times 3 \times 3 \times 3 = \mathbf{54}\text{ distinct models}$$
- **Comparative Evaluation Workload:**
  - 27 CBOW models and 27 Skip-gram models.
  - Every single model is evaluated across all 19,544 analogy quadruplets in `questions-words.txt`.
  - Baseline hyperparameters remain fixed across all models for strict experimental control (`min_count=5`, `seed=42`, `workers=NUM_WORKERS`).


In [30]:
#CELL 3.1 - DEFINE PARAMETERIZED WORD2VEC TRAINING PIPELINE
import time

#configure worker threads dynamically based on available cpu cores
NUM_WORKERS = min(4, os.cpu_count() or 2)

def train_word2vec_model(
    corpus_stream: Text8Corpus,
    sg: int,
    window: int,
    vector_size: int,
    epochs: int,
    min_count: int = 5,
    workers: int = NUM_WORKERS,
    seed: int = GLOBAL_RANDOM_SEED
) -> Word2Vec:
    #record start time for wall clock performance benchmarking
    start_time = time.time()
    
    #instantiate and train word2vec model via gensim cython engine
    model = Word2Vec(
        sentences=corpus_stream,
        sg=sg,
        window=window,
        vector_size=vector_size,
        epochs=epochs,
        min_count=min_count,
        workers=workers,
        seed=seed
    )
    
    elapsed_seconds = time.time() - start_time
    architecture_name = "Skip-gram" if sg == 1 else "CBOW"
    print(
        f"Model trained successfully [{architecture_name} | "
        f"window={window} | dim={vector_size} | epochs={epochs}] "
        f"in {elapsed_seconds:.2f}s | Vocabulary: {len(model.wv):,} words"
    )
    return model


In [31]:
#CELL 3.2 - EXECUTE SMOKE TEST TO VALIDATE TRAINING PIPELINE INTEGRITY
#train single lightweight model to verify pipeline stability
print("Launching training pipeline smoke test (CBOW, window=2, dim=50, epochs=1)...")
smoke_model = train_word2vec_model(
    corpus_stream=corpus,
    sg=0,
    window=2,
    vector_size=50,
    epochs=1
)

#verify learned vector dimensions and vocabulary properties
test_token = "france"
if test_token in smoke_model.wv:
    token_vector = smoke_model.wv[test_token]
    print(f"Smoke test verification passed:")
    print(f"- Sample token: '{test_token}'")
    print(f"- Vector shape: {token_vector.shape}")
    print(f"- Vector L2 norm: {np.linalg.norm(token_vector):.4f}")
    print(f"- Total learned vocabulary size: {len(smoke_model.wv):,} words")


Launching training pipeline smoke test (CBOW, window=2, dim=50, epochs=1)...
Model trained successfully [CBOW | window=2 | dim=50 | epochs=1] in 49.92s | Vocabulary: 71,290 words
Smoke test verification passed:
- Sample token: 'france'
- Vector shape: (50,)
- Vector L2 norm: 6.1088
- Total learned vocabulary size: 71,290 words


# 4. Vector Algebra and Statistical Evaluation Methodology

In this section, I implement the analogy evaluation pipeline grounded in vector space algebra and compute consolidated statistical metrics.

### Analogy Problem Formulation
For each analogy quadruplet of words $w_A : w_B :: w_C : w_D$ (for example, *Paris : France :: Berlin : Germany*), the model is required to identify the target concept $w_D$ given the premise tokens $w_A, w_B, w_C$.

In accordance with the assignment handout, I evaluate the resultant vector $\vec{R}$ using the vector algebra formula:
$$\vec{R} = \vec{v}_B + \vec{v}_A - \vec{v}_C = \vec{v}(\text{France}) + \vec{v}(\text{Paris}) - \vec{v}(\text{Berlin})$$

### Cosine Distance Metric and Intuitive Alignment Mapping ([-1, 1])
To quantify the angular separation between the computed resultant vector $\vec{R}$ and the true target word representation $\vec{v}_D = \vec{v}(\text{Germany})$, I first compute the classical cosine distance mandated by the assignment:
$$d(\vec{R}, \vec{v}_D) = 1 - \frac{\vec{R} \cdot \vec{v}_D}{\|\vec{R}\|_2 \|\vec{v}_D\|_2} \quad \in [0, 2]$$

While the $[0, 2]$ interval serves as a valid loss metric where lower is better, I find it significantly more intuitive to interpret vector relationships on a normalized directional alignment scale $s \in [-1, 1]$. Therefore, I map the cosine distance via the linear transformation:
$$s = 1 - d(\vec{R}, \vec{v}_D) = \frac{\vec{R} \cdot \vec{v}_D}{\|\vec{R}\|_2 \|\vec{v}_D\|_2} \quad \in [-1, 1]$$

Under this mapped alignment metric:
- **$s = +1.0$ (Perfect Alignment):** Vectors point in the exact same direction ($0^\circ$), indicating an optimal analogy solution.
- **$s = 0.0$ (Orthogonality):** Vectors are perpendicular ($90^\circ$), indicating zero directional correlation.
- **$s = -1.0$ (Diametric Opposition):** Vectors point in directly opposite directions ($180^\circ$).

This linear mapping preserves the underlying geometric relationships and statistical variance ($\sigma(s) = \sigma(d)$) without numerical distortion. In all subsequent comparative analyses, tables, and visualization charts, I adopt this $[-1, 1]$ alignment scale as the primary evaluative metric (higher is better), while preserving the raw cosine distance for formal compliance.

### Statistical Aggregation Protocol
To evaluate model performance with statistical rigor across the benchmark, I compute:
1. **Global Alignment Mean ($\overline{S}_{\text{global}}$) and Distance Mean ($\overline{D}_{\text{global}}$):** Central tendency across all covered analogy questions.
2. **Global Standard Deviation ($\sigma_{\text{global}}$):** Dispersion and stability of individual question scores.
3. **Stratified Category Metrics ($\overline{S}_c, \sigma_c$):** Individual mean alignment and standard deviation across each of the 14 semantic and syntactic categories.
4. **Stratified Block Metrics:** Macro-average alignment comparing Semantic vs. Syntactic analogy blocks.
5. **Inter-Category Dispersion:** Standard deviation across category means ($\text{std}(\{\overline{S}_1, \dots, \overline{S}_{14}\})$), measuring cross-domain consistency.


In [32]:
#CELL 4.1 - DEFINE VECTORIZED ANALOGY EVALUATION FUNCTION
#vectorized numpy implementation for analogy evaluation with dual metrics
def evaluate_word2vec_analogies(
    model: Word2Vec,
    df_benchmark: pd.DataFrame
) -> Dict[str, Any]:
    #extract keyed vectors
    kv = model.wv
    
    #dynamically identify quadruplets with full vocabulary coverage
    mask_in_vocab = df_benchmark.apply(
        lambda r: (
            r["word_a"] in kv and
            r["word_b"] in kv and
            r["word_c"] in kv and
            r["word_d"] in kv
        ),
        axis=1
    )
    
    df_eval = df_benchmark[mask_in_vocab].copy()
    valid_count = int(len(df_eval))
    total_count = int(len(df_benchmark))
    coverage_ratio = float(valid_count / total_count) if total_count > 0 else 0.0
    
    if valid_count == 0:
        return {
            "mean_alignment_global": float("nan"),
            "std_alignment_global": float("nan"),
            "mean_cosine_distance_global": float("nan"),
            "std_cosine_distance_global": float("nan"),
            "mean_semantic_alignment": float("nan"),
            "mean_syntactic_alignment": float("nan"),
            "mean_semantic_distance": float("nan"),
            "mean_syntactic_distance": float("nan"),
            "category_std_dispersion": float("nan"),
            "evaluated_quadruplets": 0,
            "total_quadruplets": total_count,
            "coverage_ratio": 0.0,
            "category_metrics": {}
        }
    
    #extract vector coordinates as float32 matrices
    #mapping word_a to v_a, word_b to v_b, word_c to v_c, word_d to v_d
    v_a = np.array([kv[w] for w in df_eval["word_a"]], dtype=np.float32)
    v_b = np.array([kv[w] for w in df_eval["word_b"]], dtype=np.float32)
    v_c = np.array([kv[w] for w in df_eval["word_c"]], dtype=np.float32)
    v_d = np.array([kv[w] for w in df_eval["word_d"]], dtype=np.float32)
    
    #vector algebra formula from assignment handout: R = v_B + v_A - v_C
    vec_r = v_b + v_a - v_c
    
    #compute inner products and norms
    norm_r = np.linalg.norm(vec_r, axis=1)
    norm_d = np.linalg.norm(v_d, axis=1)
    dot_products = np.sum(vec_r * v_d, axis=1)
    
    denominators = norm_r * norm_d
    denominators = np.where(denominators == 0, 1e-12, denominators)
    
    #cosine similarity mapped alignment in [-1, 1] where 1 is perfect
    alignment_scores = np.clip(dot_products / denominators, -1.0, 1.0)
    
    #raw cosine distance in [0, 2] where 0 is perfect: d = 1 - s
    cosine_dist = np.clip(1.0 - alignment_scores, 0.0, 2.0)
    
    df_eval["alignment_score"] = alignment_scores
    df_eval["cosine_distance"] = cosine_dist
    
    #compute global metrics
    global_mean_alignment = float(np.mean(alignment_scores))
    global_std_alignment = float(np.std(alignment_scores))
    global_mean_distance = float(np.mean(cosine_dist))
    global_std_distance = float(np.std(cosine_dist))
    
    #compute stratified metrics by section (semantic vs syntactic)
    semantic_mask = df_eval["section_type"] == "semantic"
    syntactic_mask = df_eval["section_type"] == "syntactic"
    
    mean_sem_align = float(df_eval.loc[semantic_mask, "alignment_score"].mean()) if semantic_mask.any() else float("nan")
    mean_syn_align = float(df_eval.loc[syntactic_mask, "alignment_score"].mean()) if syntactic_mask.any() else float("nan")
    mean_sem_dist = float(df_eval.loc[semantic_mask, "cosine_distance"].mean()) if semantic_mask.any() else float("nan")
    mean_syn_dist = float(df_eval.loc[syntactic_mask, "cosine_distance"].mean()) if syntactic_mask.any() else float("nan")
    
    #compute stratified metrics across 14 individual categories
    category_summary = df_eval.groupby("category").agg(
        mean_alignment=("alignment_score", "mean"),
        std_alignment=("alignment_score", "std"),
        mean_distance=("cosine_distance", "mean"),
        std_distance=("cosine_distance", "std"),
        count=("alignment_score", "count")
    ).to_dict(orient="index")
    
    category_align_means = [vals["mean_alignment"] for vals in category_summary.values() if not np.isnan(vals["mean_alignment"])]
    category_std_dispersion = float(np.std(category_align_means)) if category_align_means else float("nan")
    
    return {
        "mean_alignment_global": global_mean_alignment,
        "std_alignment_global": global_std_alignment,
        "mean_cosine_distance_global": global_mean_distance,
        "std_cosine_distance_global": global_std_distance,
        "mean_semantic_alignment": mean_sem_align,
        "mean_syntactic_alignment": mean_syn_align,
        "mean_semantic_distance": mean_sem_dist,
        "mean_syntactic_distance": mean_syn_dist,
        "category_std_dispersion": category_std_dispersion,
        "evaluated_quadruplets": valid_count,
        "total_quadruplets": total_count,
        "coverage_ratio": coverage_ratio,
        "category_metrics": category_summary,
        "evaluated_df": df_eval[["category", "section_type", "word_a", "word_b", "word_c", "word_d", "alignment_score", "cosine_distance"]]
    }


In [33]:
#CELL 4.2 - TEST EVALUATION PIPELINE ON SMOKE TEST MODEL
#evaluate smoke test model on analogy benchmark with alignment mapping
smoke_eval_results = evaluate_word2vec_analogies(smoke_model, df_analogies)

print("Smoke Model Analogy Evaluation Results:")
print(f"- Evaluated Quadruplets: {smoke_eval_results['evaluated_quadruplets']:,} / {smoke_eval_results['total_quadruplets']:,} ({smoke_eval_results['coverage_ratio']*100:.2f}% coverage)")
print(f"- Global Alignment Score [-1 to +1, higher is better]: {smoke_eval_results['mean_alignment_global']:.4f} (std: {smoke_eval_results['std_alignment_global']:.4f})")
print(f"- Global Cosine Distance [0 to 2, lower is better]:    {smoke_eval_results['mean_cosine_distance_global']:.4f} (std: {smoke_eval_results['std_cosine_distance_global']:.4f})")
print(f"- Semantic Block Alignment Score: {smoke_eval_results['mean_semantic_alignment']:.4f}")
print(f"- Syntactic Block Alignment Score: {smoke_eval_results['mean_syntactic_alignment']:.4f}")
print(f"- Inter-Category Dispersion (std of alignment): {smoke_eval_results['category_std_dispersion']:.4f}")

#tabulate category breakdown for inspection
df_smoke_categories = pd.DataFrame([
    {
        "category": cat,
        "mean_alignment": vals["mean_alignment"],
        "std_alignment": vals["std_alignment"],
        "mean_distance": vals["mean_distance"],
        "quadruplet_count": vals["count"]
    }
    for cat, vals in smoke_eval_results["category_metrics"].items()
]).sort_values(by="mean_alignment", ascending=False)

print("\nCategory-Level Alignment Breakdown (Top 5 Best vs Top 5 Worst):")
display(pd.concat([df_smoke_categories.head(5), df_smoke_categories.tail(5)]))


Smoke Model Analogy Evaluation Results:
- Evaluated Quadruplets: 17,827 / 19,544 (91.21% coverage)
- Global Alignment Score [-1 to +1, higher is better]: 0.3798 (std: 0.4866)
- Global Cosine Distance [0 to 2, lower is better]:    0.6202 (std: 0.4866)
- Semantic Block Alignment Score: 0.6474
- Syntactic Block Alignment Score: 0.1892
- Inter-Category Dispersion (std of alignment): 0.2546

Category-Level Alignment Breakdown (Top 5 Best vs Top 5 Worst):


,category,mean_alignment,std_alignment,mean_distance,quadruplet_count
2,city-in-state,0.791918,0.252572,0.208082,2330
0,capital-common-countries,0.723515,0.296065,0.276485,506
1,capital-world,0.693069,0.389821,0.306931,3564
10,gram6-nationality-adjective,0.384538,0.399354,0.615462,1521
11,gram7-past-tense,0.248008,0.450275,0.751992,1560
9,gram5-present-participle,0.136003,0.456379,0.863997,1056
13,gram9-plural-verbs,0.078716,0.419020,0.921284,870
4,family,0.057919,0.622566,0.942081,420
6,gram2-opposite,0.050905,0.456336,0.949095,756
8,gram4-superlative,0.019955,0.318404,0.980045,992


# 5. Controlled Empirical Experiments and Systematic Grid Execution

In this section, I execute the systematic Cartesian grid search across all 54 hyperparameter configurations, recording execution runtimes and evaluation metrics to disk.

### Grid Search Topology (54 Models)
The empirical evaluation explores the Cartesian product across all defined hyperparameter axes:
$$\mathcal{G} = \{0, 1\} \times \{2, 5, 10\} \times \{50, 100, 300\} \times \{1, 5, 10\} \implies 2 \times 3 \times 3 \times 3 = \mathbf{54}\text{ configurations}$$

### Experimental Protocol and Execution Tracking
1. **Model Parameterization:** Each combination trains Word2Vec on the streaming `text8` corpus with fixed baseline controls (`min_count=5`, `seed=42`, `workers=NUM_WORKERS`).
2. **Metric Logging:** Immediately following training, each model is evaluated on the 19,544 analogy questions, logging training duration, global alignment score, cosine distance, and stratified category metrics into `experiment_results.csv`.
3. **Comparative Leaderboard:** Once evaluated, models are ranked to identify the empirical configuration that maximizes analogy alignment.


In [ ]:
#CELL 5.1 - CONFIGURE AND EXECUTE PARAMETERIZED GRID SEARCH WITH DYNAMIC EXECUTION TRACKING
#define parameterized hyperparameter search vectors
GRID_ARCHITECTURES = [0, 1]          #0: cbow, 1: skip-gram
GRID_WINDOWS = [2, 5, 10]            #context window sizes
GRID_VECTOR_SIZES = [50, 100, 300]   #embedding dimensions
GRID_EPOCHS = [1, 5, 10]             #training epochs

#resolve persistent output directory across environments
if Path("TP1/outputs").exists():
    OUTPUT_DIR = Path("TP1/outputs")
elif Path("outputs").exists():
    OUTPUT_DIR = Path("outputs")
elif Path("../outputs").exists():
    OUTPUT_DIR = Path("../outputs")
else:
    OUTPUT_DIR = Path("TP1/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV_PATH = OUTPUT_DIR / "experiment_results.csv"

#helper function to format duration in human-readable units
def format_duration(seconds: float) -> str:
    total_sec = max(0, int(round(seconds)))
    hours = total_sec // 3600
    minutes = (total_sec % 3600) // 60
    secs = total_sec % 60
    if hours > 0:
        return f"{hours}h {minutes:02d}m {secs:02d}s"
    return f"{minutes:02d}m {secs:02d}s"

#capture hardware profile from centralized configuration
hw_info = get_hardware_profile()
workers_to_use = hw_info["workers_configured"]

#build parameterized cartesian grid search space
grid_configurations: List[Dict[str, Any]] = []
model_counter = 1
for sg in GRID_ARCHITECTURES:
    for window in GRID_WINDOWS:
        for vector_size in GRID_VECTOR_SIZES:
            for epochs in GRID_EPOCHS:
                grid_configurations.append({
                    "model_id": f"M{model_counter:02d}_{'SG' if sg == 1 else 'CBOW'}_w{window}_d{vector_size}_e{epochs}",
                    "sg": sg,
                    "architecture": "Skip-gram" if sg == 1 else "CBOW",
                    "window": window,
                    "vector_size": vector_size,
                    "epochs": epochs
                })
                model_counter += 1

total_models = len(grid_configurations)

#load existing results to support incremental execution
completed_configs: Set[Tuple[int, int, int, int]] = set()
if RESULTS_CSV_PATH.exists():
    try:
        df_existing = pd.read_csv(RESULTS_CSV_PATH)
        for _, row in df_existing.iterrows():
            completed_configs.add((
                int(row["sg"]),
                int(row["window"]),
                int(row["vector_size"]),
                int(row["epochs"])
            ))
    except Exception as e:
        pass

print(f"Grid Space Configured: {total_models} total models")
print(f"Already completed in previous runs: {len(completed_configs)} / {total_models}")
print(f"Remaining to execute: {total_models - len(completed_configs)} models")
print(f"Active Hardware: {hw_info['cpu_model']} | Threads: {workers_to_use} workers")

#resolve text8 streaming corpus reference
if "corpus" in globals():
    text8_stream = corpus
elif "text8_corpus" in globals():
    text8_stream = text8_corpus
else:
    text8_stream = Text8Corpus(str(TEXT8_EXTRACTED_PATH))

session_start_time = time.time()
session_models_executed = 0
session_time_spent = 0.0

#execute systematic grid search
for idx, cfg in enumerate(grid_configurations, 1):
    config_tuple = (cfg["sg"], cfg["window"], cfg["vector_size"], cfg["epochs"])
    
    #skip previously computed models
    if config_tuple in completed_configs:
        print(f"[{idx:02d}/{total_models:02d}] SKIPPED (already logged): {cfg['model_id']}")
        continue
    
    #track wall-clock time and cpu process time
    t_step_start = time.time()
    t_cpu_start = time.process_time()
    
    #train model
    model = train_word2vec_model(
        corpus_stream=text8_stream,
        sg=cfg["sg"],
        window=cfg["window"],
        vector_size=cfg["vector_size"],
        epochs=cfg["epochs"],
        min_count=MIN_WORD_COUNT,
        workers=workers_to_use,
        seed=GLOBAL_RANDOM_SEED
    )
    t_train_elapsed = time.time() - t_step_start
    t_cpu_train = time.process_time() - t_cpu_start
    
    #evaluate analogies
    t_eval_start = time.time()
    eval_metrics = evaluate_word2vec_analogies(model, df_analogies)
    t_eval_elapsed = time.time() - t_eval_start
    
    step_duration = time.time() - t_step_start
    
    #measure resident set size peak memory in mb
    peak_ram_mb = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0, 2)
    
    #calculate effective processing throughput (kwords per second)
    estimated_total_tokens = 17005207 * cfg["epochs"]
    throughput_kwords_sec = round((estimated_total_tokens / 1000.0) / max(0.001, t_train_elapsed), 2)
    
    #assemble flat record for persistent dataframe serialization
    record: Dict[str, Any] = {
        "model_id": cfg["model_id"],
        "sg": cfg["sg"],
        "architecture": cfg["architecture"],
        "window": cfg["window"],
        "vector_size": cfg["vector_size"],
        "epochs": cfg["epochs"],
        "min_count": MIN_WORD_COUNT,
        "workers": workers_to_use,
        "hardware_cpu": hw_info["cpu_model"],
        "hardware_cpu_cores": hw_info["cpu_cores"],
        "hardware_ram_gb": hw_info["total_ram_gb"],
        "hardware_gpu": hw_info["gpu_info"],
        "train_time_sec": round(t_train_elapsed, 2),
        "eval_time_sec": round(t_eval_elapsed, 4),
        "total_time_sec": round(step_duration, 2),
        "cpu_time_sec": round(t_cpu_train, 2),
        "peak_ram_mb": peak_ram_mb,
        "throughput_kwords_sec": throughput_kwords_sec,
        "evaluated_quadruplets": eval_metrics["evaluated_quadruplets"],
        "total_quadruplets": eval_metrics["total_quadruplets"],
        "coverage_ratio": round(eval_metrics["coverage_ratio"], 4),
        "mean_alignment_global": round(eval_metrics["mean_alignment_global"], 6),
        "std_alignment_global": round(eval_metrics["std_alignment_global"], 6),
        "mean_cosine_distance_global": round(eval_metrics["mean_cosine_distance_global"], 6),
        "std_cosine_distance_global": round(eval_metrics["std_cosine_distance_global"], 6),
        "mean_semantic_alignment": round(eval_metrics["mean_semantic_alignment"], 6),
        "mean_syntactic_alignment": round(eval_metrics["mean_syntactic_alignment"], 6),
        "mean_semantic_distance": round(eval_metrics["mean_semantic_distance"], 6),
        "mean_syntactic_distance": round(eval_metrics["mean_syntactic_distance"], 6),
        "category_std_dispersion": round(eval_metrics["category_std_dispersion"], 6)
    }
    
    #append per-category metrics to record
    for cat_name, cat_vals in eval_metrics["category_metrics"].items():
        clean_cat = cat_name.replace("-", "_")
        record[f"cat_{clean_cat}_mean_alignment"] = round(cat_vals["mean_alignment"], 6)
        record[f"cat_{clean_cat}_std_alignment"] = round(cat_vals["std_alignment"], 6)
        record[f"cat_{clean_cat}_mean_distance"] = round(cat_vals["mean_distance"], 6)
    
    #persist to csv incrementally
    df_single_row = pd.DataFrame([record])
    file_exists = RESULTS_CSV_PATH.exists()
    df_single_row.to_csv(
        RESULTS_CSV_PATH,
        mode="a",
        header=not file_exists,
        index=False
    )
    completed_configs.add(config_tuple)
    
    #release model memory
    del model
    gc.collect()
    
    #update dynamic execution tracking
    session_models_executed += 1
    session_time_spent += step_duration
    avg_step_time = session_time_spent / session_models_executed
    
    total_completed = len(completed_configs)
    remaining_models = total_models - total_completed
    progress_pct = (total_completed / total_models) * 100
    
    eta_seconds = remaining_models * avg_step_time
    eta_formatted = format_duration(eta_seconds)
    session_elapsed_formatted = format_duration(time.time() - session_start_time)
    projected_finish = (datetime.now() + timedelta(seconds=eta_seconds)).strftime("%H:%M:%S")
    
    #print only execution progress metadata
    print(f"\n{'='*75}")
    print(f"[{total_completed:02d}/{total_models:02d}] ({progress_pct:5.1f}%) | Model: {cfg['model_id']}")
    print(f"Configuration: {cfg['architecture']} | window={cfg['window']} | dim={cfg['vector_size']} | epochs={cfg['epochs']} | workers={workers_to_use}")
    print(f"Step duration: {format_duration(step_duration)} | Peak RAM: {peak_ram_mb} MB | Session elapsed: {session_elapsed_formatted}")
    print(f"Remaining: {remaining_models} models | Dynamic ETA: {eta_formatted} | Projected finish: {projected_finish}")
    print(f"{'='*75}")

total_elapsed = time.time() - session_start_time
print(f"\nGrid search run completed. Session time: {format_duration(total_elapsed)}.")


Grid Space Configured: 54 total models
Already completed in previous runs: 0 / 54
Remaining to execute: 54 models
Active Hardware: Intel(R) Xeon(R) CPU @ 2.20GHz | Threads: 2 workers
Model trained successfully [CBOW | window=2 | dim=50 | epochs=1] in 50.11s | Vocabulary: 71,290 words

[01/54] (  1.9%) | Model: M01_CBOW_w2_d50_e1
Configuration: CBOW | window=2 | dim=50 | epochs=1 | workers=2
Step duration: 00m 50s | Peak RAM: 1452.88 MB | Session elapsed: 00m 51s
Remaining: 53 models | Dynamic ETA: 44m 34s | Projected finish: 20:44:44
Model trained successfully [CBOW | window=2 | dim=50 | epochs=5] in 218.29s | Vocabulary: 71,290 words

[02/54] (  3.7%) | Model: M02_CBOW_w2_d50_e5
Configuration: CBOW | window=2 | dim=50 | epochs=5 | workers=2
Step duration: 03m 39s | Peak RAM: 1453.14 MB | Session elapsed: 04m 31s
Remaining: 52 models | Dynamic ETA: 1h 56m 53s | Projected finish: 22:00:42
Model trained successfully [CBOW | window=2 | dim=50 | epochs=10] in 422.68s | Vocabulary: 71,290 w

In [ ]:
#CELL 5.2 - AUDIT EXPERIMENT RESULTS AND DISPLAY SUMMARY LEADERBOARD
#load consolidated results from persistent results file
df_results = pd.read_csv(RESULTS_CSV_PATH)

print(f"Consolidated Experiment Results Summary:")
print(f"- Total Models Completed: {len(df_results)} / 54")
print(f"- Total Compute Time Logged: {df_results['train_time_sec'].sum() / 3600:.2f} hours")
print(f"- Mean Training Time per Model: {df_results['train_time_sec'].mean():.1f} seconds")
print(f"- Peak RAM Consumption Recorded: {df_results['peak_ram_mb'].max():.2f} MB")
if "hardware_cpu" in df_results.columns:
    print(f"- Executed on: {df_results['hardware_cpu'].iloc[0]} ({df_results['hardware_cpu_cores'].iloc[0]} cores, {df_results['workers'].iloc[0]} worker threads)")

#display leaderboard sorted by global alignment score
df_leaderboard = df_results[[
    "model_id", "architecture", "window", "vector_size", "epochs",
    "mean_alignment_global", "std_alignment_global",
    "mean_semantic_alignment", "mean_syntactic_alignment",
    "train_time_sec", "peak_ram_mb", "throughput_kwords_sec"
]].sort_values(by="mean_alignment_global", ascending=False).reset_index(drop=True)

print("\nTop 5 Best Performing Models (Highest Global Alignment):")
display(df_leaderboard.head(5))

print("\nComputational Trade-off: Most Efficient Models (Highest Throughput):")
display(df_results[[
    "model_id", "architecture", "window", "vector_size", "epochs",
    "mean_alignment_global", "train_time_sec", "throughput_kwords_sec", "peak_ram_mb"
]].sort_values(by="throughput_kwords_sec", ascending=False).head(5).reset_index(drop=True))
